# 01 - Data Ingestion

## 1. What I am doing

This notebook performs the data ingestion stage of the Chicago Crime Type Prediction project.

The Chicago Crime dataset was extracted from the Chicago Crime public dataset available through Google BigQuery and covers the years 2024, 2025, and 2026 year-to-date.

The dataset is loaded into Databricks and stored as the persistent raw table:

`default.chicago_crime_raw`

The ingestion process includes:
- Loading the raw dataset
- Displaying sample records
- Inspecting the schema
- Recording the total number of rows and columns
- Verifying the raw table

## 2. Why I am doing it

A reproducible raw-data layer is required before data understanding and cleaning.

Keeping the original dataset as a raw table allows the team to:
- preserve the original extracted data,
- perform data-quality investigation without modifying the source,
- reproduce the cleaning process,
- provide a consistent dataset for the later EDA, feature engineering, preprocessing, and modelling stages.

The target variable for the project is `Primary Type`, which represents the crime classification to be predicted.

In [ ]:
# Load the raw Chicago Crime dataset into a Spark DataFrame
import os

raw_csv_candidates = [
    "/FileStore/tables/chicago_crime_raw.csv",
    "dbfs:/FileStore/tables/chicago_crime_raw.csv",
    "data/raw/chicago_crime_raw.csv",
    "../data/raw/chicago_crime_raw.csv",
    "../../data/raw/chicago_crime_raw.csv",
    "/Workspace/Repos/Chicago_Crime_Prediction/data/raw/chicago_crime_raw.csv",
    "d:/Projects/Chicago_Crime_Prediction/data/raw/chicago_crime_raw.csv"
]

df_raw = None
loaded_from = None

# Check if table already exists in catalog
try:
    if spark.catalog.tableExists("default.chicago_crime_raw"):
        df_raw = spark.table("default.chicago_crime_raw")
        loaded_from = "default.chicago_crime_raw (Table)"
except Exception as e:
    pass

# If not found or empty, load from raw CSV file
if df_raw is None:
    for path in raw_csv_candidates:
        try:
            df_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
            # Verify non-empty
            if len(df_raw.columns) > 1:
                loaded_from = f"Raw CSV ({path})"
                # Persist to table for downstream notebooks
                try:
                    df_raw.write.mode("overwrite").format("delta").saveAsTable("default.chicago_crime_raw")
                    print(f"Persisted table 'default.chicago_crime_raw' from {path}")
                except Exception as save_err:
                    df_raw.createOrReplaceTempView("chicago_crime_raw")
                    print(f"Created temp view 'chicago_crime_raw': {save_err}")
                break
        except Exception:
            continue

if df_raw is None:
    raise FileNotFoundError(
        "Could not locate raw Chicago crime dataset. Please ensure 'data/raw/chicago_crime_raw.csv' or 'default.chicago_crime_raw' is accessible."
    )

print(f"Raw dataset loaded successfully from: {loaded_from}")

In [ ]:
# Display the first 10 records

display(df_raw.limit(10))

In [ ]:
# Display the Spark DataFrame schema

df_raw.printSchema()

In [ ]:
# Count total number of records

row_count = df_raw.count()

print(f"Total rows: {row_count:,}")

In [ ]:
# Count total number of columns

column_count = len(df_raw.columns)

print(f"Total columns: {column_count}")

In [ ]:
# Display all column names

for i, column in enumerate(df_raw.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
# Verify that the target variable exists

target_column = "primary_type"

if target_column in df_raw.columns:
    print(f"Target column '{target_column}' found.")
else:
    print(f"WARNING: Target column '{target_column}' was not found.")

In [ ]:
from pyspark.sql import functions as F

display(
    df_raw.groupBy("year")
          .count()
          .orderBy("year")
)

In [ ]:
# Verify that the persistent raw table exists

spark.sql("""
    SHOW TABLES IN default
""").show(truncate=False)

In [ ]:
print("=" * 60)
print("CHICAGO CRIME DATA - INGESTION SUMMARY")
print("=" * 60)

print(f"Source table       : default.chicago_crime_raw")
print(f"Total rows         : {row_count:,}")
print(f"Total columns      : {column_count}")
print(f"Target variable    : {target_column}")
print(f"Dataset period     : 2024, 2025, and 2026 year-to-date")
print("=" * 60)

## 4. Findings / Conclusion

The Chicago Crime dataset was successfully ingested into Databricks and is available as the persistent raw table `default.chicago_crime_raw`.

The ingestion validation confirmed:

- The dataset contains 22 columns.
- The target variable is `primary_type`.
- The `date` and `updated_on` fields are stored as timestamps.
- `latitude` and `longitude` are stored as numeric values.
- `arrest` and `domestic` are stored as Boolean values.
- The raw table is accessible through Spark.
- The extracted dataset covers 2024, 2025, and 2026 year-to-date.

### Dataset coverage

The extracted dataset contains 662,472 records across the selected period:

- 2024: 259,631 records
- 2025: 238,083 records
- 2026 year-to-date: 164,758 records

The 2026 records represent year-to-date data rather than a complete calendar year. Therefore, comparisons involving 2026 should account for the incomplete observation period.

No data-cleaning transformations were applied during ingestion. The raw table is preserved as the starting point for the data-understanding and cleaning stages.